In [1]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import bm25_weight, tfidf_weight
from implicit.lmf import LogisticMatrixFactorization
from scipy.sparse import csr_matrix

/Users/owen/local/unibe/ml/spotify-recommender/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Import data

In [2]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 1, "long_term": 1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,1.00,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.98,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.96,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.94,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.92,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [3]:
df_matrix_mf = df.copy()
df_matrix_mf = df_matrix_mf[df["type"].isin(["top_track"])]
df_matrix_mf["username"] = df_matrix_mf["username"].astype("category")
df_matrix_mf["id"] = df_matrix_mf["id"].astype("category")
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]
# df_matrix_mf["affinity"] *= 100
df_matrix_mf["affinity"]

0        1.00
1        0.98
2        0.96
3        0.94
4        0.92
         ... 
12318    0.10
12319    0.08
12320    0.06
12321    0.04
12322    0.02
Name: affinity, Length: 1200, dtype: float64

In [5]:
num_users = len(df_matrix_mf["username"].unique())
num_items = len(df_matrix_mf["id"].unique())
num_users, num_items

(8, 923)

In [6]:
matrix_mf = MatrixDataset(num_users, num_items)
matrix_mf.fill_from_df(df_matrix_mf["username"].cat.codes, df_matrix_mf["id"].cat.codes, df_matrix_mf["affinity"])
R = matrix_mf.matrix
R

array([[0.  , 0.  , 0.62, ..., 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , ..., 0.  , 0.  , 0.54],
       ...,
       [0.  , 0.  , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.44, 0.6 , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , ..., 0.92, 0.  , 0.  ]])

In [7]:
def convert_to_ids(values: List[str], column: str) -> List[int]:
    """
    Gets the user ids from the usernames.
    :param usernames: The usernames.
    :return: The user ids.
    """
    return df_matrix_mf[df_matrix_mf[column].isin(values)][column].cat.codes.tolist()

def retrieve_value_from_ids(ids: List[int], column: str) -> str:
    """
    Gets the value from the ids.
    :param ids: The ids.
    :return: The value.
    """
    return df_matrix_mf[df_matrix_mf[column].cat.codes.isin(ids)][column].tolist()

def get_df_rows_from_ids(ids: List[int], column: str, search_in: pd.DataFrame) -> pd.DataFrame:
    """
    Gets the dataframe rows from the ids.
    :param ids: The ids.
    :return: The dataframe rows.
    """
    return search_in[search_in[column].cat.codes.isin(ids)]

In [8]:
# Create a matrix U that contains the index that sorts the users by their affinity
I = np.argsort(matrix_mf.matrix, axis=1)
I.shape

(8, 923)

In [9]:
def compute_alpha(matrix: np.ndarray) -> np.ndarray:
    """
    Computes the alpha matrix.
    :param matrix: The matrix.
    :param k: The number of neighbors.
    :return: The alpha matrix.
    """
    alpha = len(np.where(matrix == 0)[0]) / matrix.sum()
    return alpha

alpha = compute_alpha(matrix_mf.matrix)
print(alpha)
R *= alpha
R

13.540265635507735


array([[ 0.        ,  0.        ,  8.39496469, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  7.31174344],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 5.95771688,  8.12415938,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ..., 12.45704438,
         0.        ,  0.        ]])

In [10]:
lmf = LogisticMatrixFactorization(
    factors=50,
    regularization=0.01,
    use_gpu=False,
    iterations=100,
    num_threads=1,
    random_state=42,
    dtype=np.float32,
)

R = csr_matrix(R)
lmf.fit(R, show_progress=True)

# recommend items for a user
user = "owen"
user_id = convert_to_ids([user], "username")[0]
user_items = R[user_id]
ids, scores = lmf.recommend(user_id, R[user_id], N=10, filter_already_liked_items=False)
ids = retrieve_value_from_ids(ids, "id")

df_recommended = df_matrix_mf[df_matrix_mf["id"].isin(ids)]
df_recommended[spoti.PRETTY_PRINT_FEATURES]

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:00<00:00, 261.82it/s]


,username,artists_names,name,release_year,popularity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
1516,owen,Hamza,Free YSL,2023,62,0.907,0.577,0.0574,0.063400,0.000010,0.1580,0.459,125.981,-8.146,186253,2023,62
1528,owen,betcover!!,海豚少年 - アルバムバージョン,2019,21,0.681,0.708,0.0912,0.117000,0.011000,0.1000,0.324,100.386,-7.375,293733,2019,21
1545,owen,Cinco,G13,2022,46,0.599,0.536,0.1120,0.418000,0.000000,0.2310,0.671,105.160,-9.528,150000,2022,46
1551,owen,Dosseh,Macabre,2023,56,0.815,0.648,0.3030,0.559000,0.000000,0.1080,0.707,140.043,-6.465,208841,2023,56
1556,owen,betcover!!,海豚少年 - アルバムバージョン,2019,21,0.681,0.708,0.0912,0.117000,0.011000,0.1000,0.324,100.386,-7.375,293733,2019,21
1559,owen,Cinco,G13,2022,46,0.599,0.536,0.1120,0.418000,0.000000,0.2310,0.671,105.160,-9.528,150000,2022,46
1561,owen,Dosseh,Macabre,2023,56,0.815,0.648,0.3030,0.559000,0.000000,0.1080,0.707,140.043,-6.465,208841,2023,56
1568,owen,Saez,J'accuse,2010,54,0.743,0.712,0.0624,0.075900,0.000000,0.0785,0.884,107.013,-5.018,268532,2010,54
1569,owen,Saez,No Place For Us,2002,26,0.533,0.662,0.0586,0.017400,0.000003,0.2180,0.727,154.493,-8.137,216053,2002,26
1570,owen,Zola,COEUR DE ICE (feat. Damso),2023,69,0.851,0.567,0.2310,0.215000,0.000000,0.0998,0.446,119.969,-8.074,192213,2023,69
